# TP 2 : détecter et compter la mouche de l'olive dans les pièges

**Objectif.** La mouche de l'olive (*Bactrocera oleae*) est le principal ravageur des
oliveraies méditerranéennes. On la surveille avec des pièges McPhail. Le nombre de
mouches capturées par semaine décide du traitement. Vous allez entraîner un
**détecteur** (YOLO) qui encadre chaque insecte, distingue femelles, mâles et pyrales,
puis **compte** les mouches.

**Différence avec le TP 1.** On ne classe plus une image entière. On localise
**plusieurs petits objets** dans chaque image, avec une boîte et une classe chacun.

**Ce que vous allez apprendre.**

1. Lire des annotations COCO, regarder la taille des objets.
2. Convertir au format YOLO (exercice de géométrie des boîtes).
3. Entraîner YOLO11, lire précision, rappel, mAP.
4. Passer de la détection au **comptage**, la mesure qui compte pour l'agriculteur,
   avec un seuil choisi sur la validation.
5. Tester sur d'autres pièges (Grèce) : généralisation.

**Comment travailler.** Enregistrez d'abord ce notebook sous un autre nom. Les
cellules « À FAIRE » sont à compléter, les ❓ vont dans le compte rendu, le corrigé est
dans `tp/corriges/`.

**Données** (une fois, depuis la racine du projet) :

```bash
uv run agridrone data download olive_fly_smartmcphail olive_fly_dirt_coco olive_fly_digital_mcphail
```

Licence : Ultralytics (YOLO) est sous AGPL-3.0. C'est sans problème pour ce projet
académique.

## 0. Mise en route

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch

# Racine du projet : le dossier qui contient pyproject.toml (le notebook peut être lancé
# depuis tp/ ou tp/corriges/).
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# Données extraites par `agridrone data download`. Si vous les avez mises sur un autre
# disque (configs/local.yaml), remplacez par votre chemin, ex. Path("D:/agridrone/interim").
DATA = ROOT / "data" / "interim"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
# Sous Windows et macOS, les DataLoader à plusieurs processus ne fonctionnent pas avec des
# classes définies dans un notebook : on charge les images dans le processus principal.
NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0
# Sans GPU NVIDIA (CPU, ou GPU de Mac), entraînement raccourci.
MODE_RAPIDE = DEVICE != "cuda"
print(f"Projet : {ROOT}\nMatériel : {DEVICE} ; MODE_RAPIDE = {MODE_RAPIDE}")

import json
import shutil
from collections import defaultdict

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

OUT = ROOT / "runs" / "tp2"
OUT.mkdir(parents=True, exist_ok=True)
YOLO_DIR = ROOT / "data" / "processed" / "tp2_mouche"

# Sur CPU, environ 30 s par époque en 640 px (CPU 4 cœurs), soit une demi-heure.
EPOCHS = 60 if MODE_RAPIDE else 80
IMGSZ = 640 if MODE_RAPIDE else 1024    # taille (plus grand côté) des images vues par le réseau
BATCH = 8 if MODE_RAPIDE else 16
DEVICE_YOLO = 0 if DEVICE == "cuda" else DEVICE
print(f"EPOCHS={EPOCHS}, IMGSZ={IMGSZ}")

## 1. Les données : SmartMcPhail

209 photos prises dans des pièges McPhail connectés, en oliveraies commerciales.
Chaque insecte est encadré et étiqueté `Female`, `Male` (mouche de l'olive) ou
`Glyphode` (pyrale du jasmin, autre ravageur). L'autrice fournit un découpage
train / valid / test, et aucune photo ne se trouve dans deux découpages.

Format d'annotation COCO : un fichier JSON avec la liste des `images`, des
`annotations` (boîte `bbox = [x, y, largeur, hauteur]` en pixels, coin haut-gauche) et
des `categories`.

In [ ]:
SMART = DATA / "olive_fly_smartmcphail" / "FinalDataset.coco" / "FinalDataset.coco"

def lire_coco(fichier):
    """Une ligne par boîte : image, taille de l'image, classe, boîte COCO."""
    d = json.loads(Path(fichier).read_text(encoding="utf-8"))
    categories = {c["id"]: c["name"] for c in d["categories"]}
    images = {im["id"]: im for im in d["images"]}
    lignes = [{
        "image": images[a["image_id"]]["file_name"],
        "W": images[a["image_id"]]["width"], "H": images[a["image_id"]]["height"],
        "classe": categories[a["category_id"]],
        "x": a["bbox"][0], "y": a["bbox"][1], "w": a["bbox"][2], "h": a["bbox"][3],
    } for a in d["annotations"]]
    return pd.DataFrame(lignes), [im["file_name"] for im in d["images"]]

boites = {}
for split in ("train", "valid", "test"):
    boites[split], _ = lire_coco(SMART / split / "_annotations.coco.json")
    boites[split]["split"] = split
toutes = pd.concat(boites.values(), ignore_index=True)
pd.crosstab(toutes["classe"], toutes["split"], margins=True)

### Exercice 1.1 : quelle taille font les insectes ?

Ajoutez une colonne `taille_rel` = largeur de la boîte / plus grand côté de l'image
(`max(W, H)` : la plupart des photos sont en portrait). Tracez son histogramme. Puis,
pour `IMGSZ` = 640, 1024 et 1280, calculez la largeur d'un insecte en pixels (médiane
et 10e centile) une fois l'image réduite à `IMGSZ`.

❓ Un détecteur a du mal en dessous d'environ 10 pixels. Quelle valeur d'`IMGSZ`
conseillez-vous, et à quel coût de calcul ?

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
# toutes["taille_rel"] = ...
# toutes["taille_rel"].plot.hist(...)
# for taille in (640, 1024, 1280): px = toutes["taille_rel"] * taille ; print(px.median(), px.quantile(0.1))

### Exercice 1.2 : afficher une image avec ses boîtes

Écrivez `afficher(split, image)`, qui dessine l'image et ses boîtes : un rectangle
`patches.Rectangle((x, y), w, h)` par boîte, avec une couleur par classe.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
COULEURS = {"Female": "#e4572e", "Male": "#17bebb", "Glyphode": "#ffc914"}

def afficher(split, image):
    ...

afficher("train", boites["train"]["image"].iloc[0])

❓ Comment distinguez-vous une femelle d'un mâle ? (Indice : l'ovipositeur au bout de
l'abdomen.) Ce détail est-il visible à la résolution de travail du réseau ?

## 2. Convertir au format YOLO

YOLO attend, pour chaque image, un fichier `.txt` avec une ligne par objet :

```
classe x_centre y_centre largeur hauteur
```

Les quatre nombres sont **normalisés** entre 0 et 1 par la taille de l'image. COCO
donne `x, y` = coin haut-gauche, en pixels.

### Exercice 2.1

Complétez `coco_vers_yolo`. Copiez chaque image dans `<sortie>/images/<split>` et
écrivez ses étiquettes dans `<sortie>/labels/<split>`, dans un fichier `.txt` **de
même nom** que l'image (`photo.jpg` → `photo.txt`) : c'est ainsi que YOLO associe
image et étiquettes. Les `category_id` se traduisent en noms avec `d["categories"]`,
puis en numéros de classe avec `correspondance`. La catégorie `Gliphode-Male-Female`
est une super-catégorie vide de Roboflow : ignorez toute catégorie absente de
`correspondance`.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
CLASSES = ["femelle", "male", "pyrale"]
COCO_VERS_ID = {"Female": 0, "Male": 1, "Glyphode": 2}

def coco_vers_yolo(json_coco, dossier_images, split, sortie=YOLO_DIR, correspondance=COCO_VERS_ID):
    d = json.loads(Path(json_coco).read_text(encoding="utf-8"))
    ...  # pour chaque image : lignes "id xc yc w h" normalisées, fichier .txt, copie de l'image

for split_coco, split_yolo in [("train", "train"), ("valid", "val"), ("test", "test")]:
    coco_vers_yolo(SMART / split_coco / "_annotations.coco.json", SMART / split_coco, split_yolo)

In [ ]:
# Fourni : fichier de description du jeu pour YOLO, et vérification.
def ecrire_yaml(dossier, noms, train="images/train", val="images/val", test="images/test"):
    """Ultralytics exige les clés train et val, même pour une simple évaluation."""
    fichier = dossier / "data.yaml"
    fichier.write_text(f"path: {dossier.as_posix()}\ntrain: {train}\nval: {val}\ntest: {test}\n"
                       + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(noms)), encoding="utf-8")
    return fichier

DATA_YAML = ecrire_yaml(YOLO_DIR, CLASSES)
for split in ("train", "val", "test"):
    images = sorted((YOLO_DIR / "images" / split).iterdir())
    sans_txt = [p.name for p in images if not (YOLO_DIR / "labels" / split / f"{p.stem}.txt").exists()]
    n_box = sum(len([l for l in f.read_text().split("\n") if l]) for f in (YOLO_DIR / "labels" / split).glob("*.txt"))
    print(f"{split:5s} : {len(images)} images, {n_box} boîtes, {len(sans_txt)} image(s) sans fichier d'étiquettes")
print(DATA_YAML.read_text())

❓ Vérifiez une conversion à la main. Reprenez la première boîte d'une image, en
pixels, et retrouvez-la à partir de la ligne YOLO.

## 3. Entraîner YOLO11

`yolo11n.pt` est la plus petite version (2,6 M de paramètres). Elle est pré-entraînée
sur le jeu d'images COCO (80 classes d'objets du quotidien, sans insecte), celui qui a
donné son nom au format d'annotation vu en partie 1. Ultralytics s'occupe de la
boucle, des augmentations (mosaïque de 4 images, couleurs, échelles) et des courbes.
La mosaïque est coupée pendant les dernières époques (`close_mosaic`), pour finir sur
des images naturelles.

Durée indicative : environ 30 s par époque sur un CPU 4 cœurs en 640 px, soit une
demi-heure en `MODE_RAPIDE`. Sur GPU, 80 époques en 1024 px prennent de l'ordre d'une
demi-heure (non mesuré ici).

In [ ]:
from ultralytics import YOLO

modele = YOLO("yolo11n.pt")   # téléchargé automatiquement la première fois
resultats = modele.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE_YOLO,
    close_mosaic=min(10, EPOCHS // 5), seed=SEED, workers=NUM_WORKERS,
    project=str(OUT), name="mouche", exist_ok=True, plots=True, verbose=False,
)
MEILLEUR = OUT / "mouche" / "weights" / "best.pt"
courbes = pd.read_csv(OUT / "mouche" / "results.csv")
courbes.plot(x="epoch", y=["metrics/mAP50(B)", "metrics/mAP50-95(B)"], figsize=(5, 3))
plt.title("Validation au fil des époques"); plt.show()
print(MEILLEUR)

## 4. Évaluer la détection

- **IoU** (intersection sur union) : aire commune à la boîte prédite et à la vraie
  boîte, divisée par l'aire de leur réunion (1 = superposition parfaite). Une
  prédiction est juste si son IoU avec une vraie boîte de même classe est ≥ 0,5.
- **Précision** : part des boîtes prédites qui sont justes. **Rappel** : part des vrais
  insectes trouvés. Ultralytics les donne au seuil de confiance qui maximise le F1.
- **AP** (*average precision*) : aire sous la courbe précision-rappel, obtenue en
  faisant varier le seuil de confiance. **mAP50** : moyenne des AP des classes, à
  IoU ≥ 0,5. **mAP50-95** : moyenne des mAP pour des IoU exigés de 0,50 à 0,95 ; elle
  récompense les boîtes bien ajustées.

### Exercice 4.1

Évaluez `MEILLEUR` sur le découpage `test`, puis affichez précision, rappel, mAP50 et
mAP50-95 par classe. Le test ne contient que 7 pyrales : leurs scores sont très
incertains.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
# metriques = YOLO(MEILLEUR).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE_YOLO,
#                                plots=True, project=str(OUT), name="test", exist_ok=True)
# metriques.box.p, metriques.box.r, metriques.box.ap50, metriques.box.ap  (un élément par classe)

❓ Ouvrez `runs/tp2/test/confusion_matrix.png` (à la racine du projet). Attention :
Ultralytics place les vraies classes en colonnes (axe « True ») et les prédictions en
lignes, à l'inverse de la matrice du TP 1. Le détecteur confond-il surtout femelles et
mâles, ou rate-t-il des insectes (ligne « background » : insectes réels non
détectés) ? Que compte la colonne « background » ?

## 5. Du détecteur au comptage

L'agriculteur n'a pas besoin de boîtes. Il a besoin du **nombre de mouches** par
piège. On compte les détections de femelles et de mâles dont la confiance dépasse un
seuil `CONF`. Ce seuil est un réglage : on le choisit sur la **validation**, comme le
seuil « incertain » du TP 1, puis on l'applique **une seule fois** au test.

La fonction `detections` (fournie) passe chaque image une fois dans le détecteur, avec
un seuil très bas, et garde classes et confiances : on peut ensuite compter pour
n'importe quel seuil sans tout recalculer.

In [ ]:
detecteur = YOLO(MEILLEUR)

def detections(images):
    """{nom d'image: (classes, confiances)} pour une liste de chemins d'images."""
    sortie = {}
    for image in images:
        r = detecteur.predict(str(image), conf=0.01, imgsz=IMGSZ, device=DEVICE_YOLO,
                              max_det=1000, verbose=False)[0]
        sortie[Path(image).name] = (r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy())
    return sortie

def vrais_comptes(split):
    """{nom d'image: nombre de mouches (femelles + mâles) annotées}."""
    comptes = {}
    for image in (YOLO_DIR / "images" / split).iterdir():
        lignes = (YOLO_DIR / "labels" / split / f"{image.stem}.txt").read_text().split("\n")
        comptes[image.name] = sum(1 for l in lignes if l and int(l.split()[0]) in (0, 1))
    return comptes

dets = {split: detections(sorted((YOLO_DIR / "images" / split).iterdir())) for split in ("val", "test")}
vrais = {split: vrais_comptes(split) for split in ("val", "test")}

### Exercice 5.1

Écrivez `compter(dets, vrais, conf)`, qui renvoie un tableau image / vrai / prédit (on
compte les classes 0 et 1 dont la confiance est ≥ `conf`). Sur la **validation**,
balayez `conf` de 0,05 à 0,6 et retenez celui qui minimise l'erreur absolue moyenne.
Appliquez-le au **test** : erreur absolue moyenne, totaux, nuage prédit contre vrai.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
def compter(dets, vrais, conf):
    ...  # DataFrame avec colonnes image, vrai, predit

# balayage sur dets["val"] -> CONF ; puis comptes_test = compter(dets["test"], vrais["test"], CONF)

❓ Sur la validation, comment évoluent l'erreur de comptage et le total quand `CONF`
passe de 0,05 à 0,6 ? Quel seuil retenez-vous pour décider d'un traitement, et que
donne-t-il sur le test ?

## 6. Le vrai test : d'autres pièges

### 6.1 DIRT (Grèce, 2018)

271 images annotées (291 dans l'archive) de pièges McPhail, avec des insectes flottant
dans le liquide. Les annotations ont une seule classe, `insect`, qui couvre **tous**
les insectes du piège, y compris d'autres espèces que la mouche de l'olive. On évalue
donc la capacité à **trouver les insectes**, sans distinguer les classes
(`single_cls=True`). Pour comparer à armes égales, réévaluez aussi le test
SmartMcPhail avec `single_cls=True`.

### Exercice 6.1

Convertissez DIRT au format YOLO avec votre `coco_vers_yolo` (annotations dans
`instances_default.json`, une seule classe : `correspondance={"insect": 0}`, sortie
dans `data/processed/tp2_dirt`, découpage `test`). Écrivez son `data.yaml` avec
`ecrire_yaml`. Évaluez ensuite le détecteur avec `single_cls=True`, sur DIRT et sur le
test SmartMcPhail.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
DIRT = DATA / "olive_fly_dirt_coco" / "DIRT" / "DIRT"
DIRT_YOLO = ROOT / "data" / "processed" / "tp2_dirt"
# coco_vers_yolo(DIRT / "instances_default.json", DIRT, "test", sortie=DIRT_YOLO, correspondance={"insect": 0})
# yaml_dirt = ecrire_yaml(DIRT_YOLO, ["insecte"], train="images/test", val="images/test")
# YOLO(MEILLEUR).val(data=..., split="test", single_cls=True, project=str(OUT), name="dirt", ...)

❓ Comparez la mAP50 mono-classe sur SmartMcPhail et sur DIRT. Le score s'effondre-t-il ?
Regardez `runs/tp2/dirt/val_batch0_labels.jpg` (vérité) et `val_batch0_pred.jpg`
(prédictions). Qu'est-ce qui change : liquide, reflets, densité d'insectes, cadrage,
autres espèces que le modèle n'a jamais vues ?

### 6.2 Piège McPhail numérique (Leucade, 2022)

125 images, sans boîtes, tirées au hasard dans six pièges. Le fichier `counts.csv`
donne pour chaque image (`image_file`) le nombre total de mouches visibles, recompté à
l'aveugle par un expert (`expert_count`). C'est exactement la mesure utile sur le
terrain. Attention à deux colonnes :
- `pipeline_count` est un incrément journalier : ne la comparez pas à vos comptages ;
- `density_bin` classe les images selon cet incrément, pas selon le contenu du piège.
  Pour étudier l'erreur selon la densité réelle, faites vos propres tranches
  d'`expert_count` (`pd.cut`).

### Exercice 6.2

Prédisez le nombre de mouches de chaque image, avec le `CONF` choisi sur la
validation. Comparez-le au comptage de l'expert : erreur absolue moyenne, nuage de
points, puis erreur par tranche de densité réelle.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
DMT = DATA / "olive_fly_digital_mcphail" / "Zenodo_DMT_validation_dataset"
experts = pd.read_csv(DMT / "counts.csv")
# dets_dmt = detections([DMT / "images" / nom for nom in experts["image_file"]])
# compter les classes 0 et 1 au-dessus de CONF, comparer à expert_count

❓ L'erreur augmente-t-elle avec le nombre de mouches dans le piège ? Pourquoi un piège
plein (insectes qui se chevauchent, abîmés) est-il plus difficile ? Ce compteur
suffirait-il pour décider d'un traitement ?

## 7. Pour aller plus loin

1. **Petits objets** : découpez chaque image en tuiles de 640 px qui se recouvrent,
   détectez sur chaque tuile, puis fusionnez les boîtes. La bibliothèque SAHI fait
   cela ; essayez-la sans modifier l'environnement du projet :
   `uv run --with sahi jupyter lab tp/`. Le rappel augmente-t-il ?
2. **Plus de données** : ajoutez le jeu croate de Zadar à l'entraînement
   (`uv run agridrone data download olive_fly_zadar`, environ 900 Mo, boîtes Pascal
   VOC). Écrivez la conversion VOC → YOLO.
3. **Un modèle plus grand** : `yolo11s.pt`. Que gagne-t-on en mAP, et que perd-on en
   vitesse ?
4. **Vers le drone** : les pièges jaunes englués peuvent être photographiés par un
   drone qui survole l'oliveraie. Quelle résolution au sol faudrait-il pour voir une
   mouche de 5 mm sur au moins 10 pixels ? (`GSD = taille / pixels`)

## Compte rendu attendu

Une à deux pages :
- effectifs et taille des insectes ;
- choix d'`IMGSZ` justifié ;
- tableau des métriques du test ;
- seuil `CONF` choisi sur la validation et erreur de comptage sur le test ;
- résultats hors source (DIRT, Leucade) ;
- vos réponses aux ❓.